# Topic 22: Range Queries

Answer many range questions quickly: prefix sums for static data, difference arrays for batches of range updates, Fenwick trees for sums with updates, and segment trees for any associative operation.

*Generated from the course by `tools/export-docs.mjs`. Run the cells from the repository root (`jupyter notebook notebooks/`) so that `dsa_course` can be imported. Interactive questions and visualizers live in the browser course: https://rizwan3659.github.io/python-dsa-course/#t/rangeq*

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

## 1. The range query problem

"What is the sum (or minimum) of positions l to r?" asked thousands of times. The right structure depends on whether the data changes, and how.

| Situation | Tool | Build | Update | Query |
|---|---|---|---|---|
| No changes, sums | Prefix sums | O(n) | O(n) rebuild | O(1) |
| Many range adds, then read once | Difference array | O(n) | O(1) | O(n) once at the end |
| Point updates + range sums | Fenwick tree | O(n log n) | O(log n) | O(log n) |
| Point updates + range min/max/gcd… | Segment tree | O(n) | O(log n) | O(log n) |

**Your turn** (answer these in the browser course or locally):

- q queries the naive way (complexity, `rq-c2`)
- When prefix sums stop working (concept, `rq-m1`)
- Static sums (choose, `rq-ch1`)
- Many range additions, one read (choose, `rq-ch2`)
- Updates and minimums interleaved (choose, `rq-ch3`)
- Range sum misses the last element (debug, `rq-d3`) · locally: `python dsa.py test rq-d3`

**Code it** (answer these in the browser course or locally):

- Range Sum Query (static) (code, `rq-static-sum`) · locally: `python dsa.py test rq-static-sum`
- 2D Region Sums (code, `rq-region-sum`) · locally: `python dsa.py test rq-region-sum`

## 2. Difference arrays

A difference array stores how much each value changes from the previous one. Adding v to a whole range changes only two of those differences.

**Visualizer:** Difference array: open https://rizwan3659.github.io/python-dsa-course/#v/diff-array

**Your turn** (answer these in the browser course or locally):

- Two range additions (predict, `rq-p1`)
- What a difference array is for (concept, `rq-m2`)
- Rebuilding from a difference array (dry run, `rq-dr1`)
- Complete the difference array (complete, `rq-b2`)
- Off the end of the difference array (debug, `rq-d2`) · locally: `python dsa.py test rq-d2`

**Code it** (answer these in the browser course or locally):

- Apply Range Additions (code, `rq-range-add`) · locally: `python dsa.py test rq-range-add`
- Flight Bookings (code, `rq-flight-bookings`) · locally: `python dsa.py test rq-flight-bookings`
- Car Pooling (code, `rq-car-pooling`) · locally: `python dsa.py test rq-car-pooling`

## 3. Fenwick trees

A Fenwick (binary indexed) tree stores partial sums whose lengths are powers of two, decided by the lowest set bit of the index. Prefix sums and updates both touch only O(log n) cells.

**Visualizer:** Fenwick tree paths: open https://rizwan3659.github.io/python-dsa-course/#v/fenwick

Query prefix(11), then switch to add(i, delta) with i = 3 and watch the climb.

**Your turn** (answer these in the browser course or locally):

- Lowest set bit (predict, `rq-p2`)
- Fenwick query path (predict, `rq-p3`)
- Fenwick update path (predict, `rq-p4`)
- Fenwick indexes start at 1 (concept, `rq-m5`)
- Fenwick prefix query (complexity, `rq-c1`)
- Fenwick prefix sum of [1..8] up to 7 (dry run, `rq-dr2`)
- Complete the Fenwick loops (complete, `rq-b1`)
- The update that went the wrong way (debug, `rq-d1`) · locally: `python dsa.py test rq-d1`

**Code it** (answer these in the browser course or locally):

- Build a Fenwick Tree (code, `rq-fenwick`) · locally: `python dsa.py test rq-fenwick`
- Range Sum Query (mutable) (code, `rq-num-array`) · locally: `python dsa.py test rq-num-array`
- Count Smaller Numbers After Self (code, `rq-count-smaller`) · locally: `python dsa.py test rq-count-smaller`

## 4. Segment trees

A segment tree is a binary tree over the array: each node stores the answer for its range. A query combines O(log n) nodes, so any associative operation works, including min and max.

**Visualizer:** Segment tree query: open https://rizwan3659.github.io/python-dsa-course/#v/segtree

**Your turn** (answer these in the browser course or locally):

- Fenwick vs segment tree (concept, `rq-m3`)
- Segment tree size (concept, `rq-m4`)

**Iterative sum segment tree (size 2n)**

In [ ]:
class SumTree:
    def __init__(self, nums):
        self.n = len(nums)
        self.t = [0] * self.n + list(nums)          # leaves at n..2n-1
        for i in range(self.n - 1, 0, -1):
            self.t[i] = self.t[2 * i] + self.t[2 * i + 1]

    def update(self, i, value):
        i += self.n
        self.t[i] = value
        while i > 1:
            i //= 2
            self.t[i] = self.t[2 * i] + self.t[2 * i + 1]

    def query(self, l, r):                          # inclusive l..r
        total, l, r = 0, l + self.n, r + self.n + 1
        while l < r:
            if l & 1:
                total += self.t[l]
                l += 1
            if r & 1:
                r -= 1
                total += self.t[r]
            l //= 2
            r //= 2
        return total

st = SumTree([5, 8, 6, 3, 2, 7, 2, 6])
print(st.query(2, 6))
st.update(3, 10)
print(st.query(2, 6))

**Code it** (answer these in the browser course or locally):

- Segment Tree for Range Minimum (code, `rq-segtree-min`) · locally: `python dsa.py test rq-segtree-min`

## Challenge: Range sums with updates: re-sum → Fenwick tree

Support update(i, value) and sum_range(l, r), mixed together, many times.

**Brute force** (O(n)): Each query sums up to n values.

In [ ]:
class NumArray:
    def __init__(self, nums):
        self.nums = list(nums)

    def update(self, i, value):
        self.nums[i] = value

    def sum_range(self, l, r):
        return sum(self.nums[l:r + 1])

**Optimized** (O(log n)): Both update and sum_range touch O(log n) tree cells.

When data changes, find a structure where each change and each question touches only a logarithmic number of stored pieces.

In [ ]:
def update(self, i, value):
    delta, self.nums[i] = value - self.nums[i], value
    i += 1
    while i < len(self.tree):
        self.tree[i] += delta
        i += i & -i

## Review

Cover the right column and answer aloud.

| Question | Answer |
|---|---|
| Static range sums? | Prefix sums: O(1) per query. |
| Range adds then one read? | Difference array: O(1) per update. |
| i & −i in a Fenwick tree? | The length of the range tree[i] covers. |
| Fenwick query / update loops? | i −= i & −i down; i += i & −i up. |
| Why 1-based Fenwick? | 0 & −0 = 0: the loop would not move. |
| Range min with updates? | Segment tree (min has no inverse). |
| Segment tree query cost? | O(log n): at most two nodes per level. |